# 19 — Sessions with automatic saving
This notebook runs without an LLM. A fixture analyzer supplies the two-task plan; real mock tools execute. It tests persistence and session behavior, not language understanding. SQLite history from steps 1–2 remains available.

In [1]:
from pathlib import Path
import sys
project_root = next((p for p in (Path.cwd(), *Path.cwd().parents)
                     if (p / "src/agent1").is_dir()), None)
if project_root is None:
    raise FileNotFoundError("Open inside Agent1.")
sys.path.insert(0, str(project_root / "src"))
from agent1.session import ChatSession


In [2]:
from types import SimpleNamespace
import numpy as np
from agent1.analysis.action_plan import ActionPlan
from agent1.persistent_agent import PersistentAgent
from agent1.conversation.context_builder import ContextBuilder
from agent1.routing.action_types import ActionType, RoutingDecision
def factory(store):
    p=ActionPlan.model_validate({'request_type':'multi_action','reason':'fixture','steps':[
        {'intent':'track_order','action':'order_service','arguments':{}},
        {'intent':'availability','action':'product_service','arguments':{'product':'backpack','color':'blue'}}]})
    class Analyzer:
        def analyze(self,**kwargs):return p.model_copy(deep=True)
        def classify_pending_reply(self,*args):return 'abandon'
    routing=RoutingDecision('track_order',1,ActionType.ORDER_SERVICE,1,1,[],False,'fixture')
    return PersistentAgent(
        ensemble_model=SimpleNamespace(classes_=np.array(['track_order']),predict_proba=lambda _:np.array([[1.0]])),
        router=SimpleNamespace(route=lambda *_:routing),analyzer=Analyzer(),
        context_builder=ContextBuilder(),checkpoint_store=store)


In [3]:
from agent1.persistence.sqlite_session_store import SQLiteSessionStore
store = SQLiteSessionStore(project_root / "data/local/conversations.sqlite3")


In [4]:
session = ChatSession(store, factory)
conversation_id = session.new()
print("Conversation:", conversation_id)
print(session.send("Track my order and check the blue backpack."))
# No manual save here: the pending turn is already saved.
restored = ChatSession(store, factory)
restored.open(conversation_id)
assert restored.agent.state.get_missing_arguments() == ["0.order_id"]
print(restored.send("ORD-123"))
# Completion is saved automatically too.
reopened = ChatSession(store, factory)
reopened.open(conversation_id)
assert not reopened.agent.state.has_pending_action()
assert sum(m.role.value == "tool" for m in reopened.agent.state.messages) == 2
print("Automatic save / reopen / complete: PASS")
for item in store.list_conversations(limit=5):
    print(item)


Conversation: chat-835dca527686400b889b4910916d32ae
For task 1 (track_order), please provide one order number, such as ORD-123.
1. Order ORD-123: shipped. Estimated delivery: 2026-09-23.
2. Mock inventory: blue backpack: 5 available.
Automatic save / reopen / complete: PASS
{'conversation_id': 'chat-835dca527686400b889b4910916d32ae', 'created_at': '2026-10-27 16:00:44', 'updated_at': '2026-10-27 16:00:44', 'preview': 'Track my order and check the blue backpack.', 'message_count': 6, 'pending': False}
{'conversation_id': 'live-resume-8c154c87dd064cd49f9d476d6405f233', 'created_at': '2026-10-27 15:22:35', 'updated_at': '2026-10-27 15:22:42', 'preview': 'Track my order and check whether the blue backpack is available.', 'message_count': 6, 'pending': False}
{'conversation_id': 'offline-resume-328afaecc3504abdb13f3c8c84c8a26c', 'created_at': '2026-10-27 15:05:45', 'updated_at': '2026-10-27 15:08:11', 'preview': 'Track my order and check whether the blue backpack is available.', 'message_co

## Abandonment is also saved

In [5]:
abandoned_id = session.new()
session.send("Track my order and check the blue backpack.")
print(session.send("Never mind"))
check = ChatSession(store, factory)
check.open(abandoned_id)
assert not check.agent.state.has_pending_action()
assert check.agent._fresh_request
print("Abandonment saved: PASS")


Okay, I have stopped that request. No order was cancelled.
Abandonment saved: PASS


## Next
For MongoDB, use notebook 20 with a running MongoDB deployment. For live conversation, paste `21_chat_setup.py` into notebook 16 with its loaded model, then run `run_chat(session)`. The fixture analyzer in this notebook is deliberately not a conversational model.